<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - ZARC Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load the CSV

In [ ]:
import pandas as pd

# Path to the CSV file
file_path = "results/zarc_test_file.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first few rows
print("✅ CSV file loaded successfully. Preview:")
display(df.head())

# Show available columns
print("\nColumns in the file:")
print(df.columns.tolist())

## **📥 Step 3: Extract analytics - function from processor_zarc_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import the ZARC extractor class
from earthdaily.agriculture.processors.processor_zarc_functions import ZARCExtractor

# Instantiate the extractor
extractor = ZARCExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config
)

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

# Define nb_days_sowing_emergence
nb_days_sowing_emergence = 20  # Example value

# Function to configure and call API for each row

extractor.setup_zarc_parameters(
        crop="OTHERS",
        nb_days_sowing_emergence=20,
        soil_type=None,
        cycle=None,
        partial_frequency= 50,
        column_mapping=column_mapping
    )

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS",
    "emergence_date": "2025-11-01",
    "nb_days_sowing_emergence": 20,
    "soil_type": " ", #Optional
    "cycle": " "      #Optional               
}


#### Test get_zarc_api

In [ ]:
print("\n--- Test: get_zarc ---")
try:
    # Invoke API
    result = extractor.get_zarc_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else str(result)[:500])

    # Format to DataFrame
    formatted_df = extractor.format_zarc_json(result)
    if formatted_df.empty:
        print("⚠️ The response does not contain 'data'. Check the parameters and the API contract.")
    else:
        print("✅ Formatted DataFrame (head):")
        display(formatted_df.head())

except Exception as e:
    print(f"❌ API call failed: {e}")


### 🗺️ process_single_entity_zARC

In [ ]:
import pandas as pd

# Single row with required query fields
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS",
    "emergence_date": "2025-11-01",   # required for query (date_emergence)
    "nb_days_sowing_emergence": 20    # required for query
})

# Call single-entity processor
single_result = extractor.process_single_entity_zarc(row)
print(single_result.get("error") or "✅ ZARC row processed.")

# Show formatted result
df_single = single_result.get("data")
display(df_single.head() if df_single is not None else df_single)


### 🗺️ process_zarc_bulk_extraction_parallel

In [ ]:
# Minimal prep: take the first 500 rows
top_entities = df.head(50)

In [ ]:
# Run parallel ZARC extraction
result = extractor.process_zarc_bulk_extraction_parallel(
    entity_list=top_entities,                 # your df (no need for nb_days column)          # optional extras (geometry in output)
    max_workers=10,
    output_path=manager.output_result_dir,
    partial_frequency=50,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="SOYBEANS",
    filter_type="include",
    merge_existing='auto',
    skip_export=False,
    prefix="zarc",
)

print("\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(f"Columns: {list(result['results_df'].columns)}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")